
<br>
Itterate over all subjects folders, convert mat files of epoched data to EpochsArray instances, combine epochs<br>
to the desired conditions and save the two epochs arrays to fif files.<br>
Analyse epochs_combined data using CSD, TFR, GFP, PSD and topo plots. <br>
Subjects foldes must contain only one mat file that contains the epoched data and one raw MEG bti recording.<br>



importations of libraries
<br>
if __name__ == "__main__":<br>
    package_path = "C:/Projects/food_meg_analyses" <br>
    import numpy as np<br>
    import numbers<br>
    import os<br>
    import mne<br>
    import glob<br>
    import warnings<br>
    from mne.time_frequency import csd_morlet, read_spectrum, read_csd, read_tfrs<br>
    import traceback<br>
    from pymatreader import read_mat<br>
    from src import config<br>
    from mat_to_epochs_conversion import convert_main_funcs, combine_epochs, create_info # using * didn't work for some reason<br>
    from analyses import *<br>
    import sys<br>
    if package_path not in sys.path:<br>
        sys.path.insert(0, package_path)<br>
   <br>
    # in case one of the modules is not installed or can not be found by python using the system variables:<br>
    # except Exception as e:<br>
    #     print("An error occured:", e)<br>
    #     traceback.print_exc()<br>
    try:<br>
        directory = config.subject_directory_pattern # directory pattern for itterating over subject folders<br>
        # itterate over all subjects folders<br>
        for folder in glob.iglob(directory): <br>
            if os.path.exists(folder): # the subject folder that contains the mat file with epoched data and the raw MEG recordings per subject<br>
                try:  <br>
                    subject_num = folder.split("SUBS_DIR\\", 1)[1]<br>
                    os.chdir(folder)<br>
                    report = mne.Report(title=f"report for {subject_num}")<br>
                    raw_info = create_info.extract_raw_info(folder)<br>
                    <br>
                    # recieves a file path to the mat file, glob.glob returns a list of all paths found with the pattern.<br>
                    # [0] for returning the first and only element in the list.<br>
                    epochs, evoked = convert_main_funcs.convert_mat_to_epochs(glob.glob(config.mat_file_path_pattern)[0], info=raw_info) <br>
                    <br>
                    # combine epochs by new conditions (new_event_ids):<br>
                    epochs_combined = combine_epochs.combine_epochs(epochs, config.event_ids, config.new_event_ids)<br>
                <br>
                    # extract conditions for csd cmputation per condition:<br>
                    conditions =  list(epochs_combined.event_id.keys())<br>
                <br>
                    # Suppress warning about wavelet length.<br>
                    warnings.simplefilter('ignore')<br>
                    for condition in conditions:<br>
                        # csd calculation post stimulus over the desired frequency range per condition, save and add to report<br>
                        csd = compute_csd.compute_csd(epochs_combined, condition, config.freq_bands, config.post_stim_time) <br>
                        <br>
                    # csd calculation of baseline over the desired frequency range, save and add to report. (calculates csd baseline for the last <br>
                    # condition in loop, we assume that all conditions have same baseline activity)<br>
                    csd_baseline = compute_csd(epochs, condition, config.freq_bands, config.baseline_time)<br>
                    <br>
                    # compute tfrs for desired contrast of conditions, over the frequencies in freqs and save:<br>
                    tfr = tfr_psd_analyses.compute_tfr_contrast(epochs=epochs, subject_num=subject_num, freqs=np.arange(8, 24, 2), con1=('pres_1', config.pres_1), <br>
                    con2=('pres_2', config.pres_2), report=report)<br>
                    tfr = tfr_psd_analyses.compute_tfr_contrast(epochs=epochs, subject_num=subject_num, freqs=np.arange(8, 24, 2), con1=('food',config.food), <br>
                    con2=('nonfood',config.nonfood), report=report)<br>
                        <br>
                except Exception as e:<br>
                    print("An error occured:", e)<br>
                    traceback.print_exc()<br>
            else:<br>
                raise FileNotFoundError<br>
    except Exception as e:<br>
        print("An error occured:", e)<br>
        traceback.print_exc()